# Fine-tune RMBG-2.0 for Line Art Removal

## BiRefNet: Alternating Freeze Strategy for Domain Adaptation

Line art (black & white anime) = different feature distribution than natural images:
- Low-level: Need to relearn edge detection for thin strokes
- Mid-level: Alternating freeze to preserve semantic while adapting
- High-level: Keep object-level features frozen

### 1. Clone project from GitHub

In [ ]:
!git clone https://github.com/hoangtung386/BRG-fine-tuning-model.git /content/fine_tune_model_BRG
%cd /content/fine_tune_model_BRG

In [ ]:
!pip install -qqq -r requirements.txt

### 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import userdata
import os

hf_token = userdata.get('HF_TOKEN')
wandb_token = userdata.get('WANDB_TOKEN')

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    !huggingface-cli login --token $hf_token
    print("HuggingFace logged in")
else:
    print("HF_TOKEN not found")

if wandb_token:
    os.environ['WANDB_API_KEY'] = wandb_token
    !wandb login
    print("W&B logged in")
else:
    print("WANDB_TOKEN not found")

### 3. Check GPU & Install dependencies

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

### 4. Setup config for Colab

In [ ]:
import sys
sys.path.insert(0, '/content/fine_tune_model_BRG')

from config import config

# Cập nhật paths cho Colab
config.DATASET_PATH = '/content/drive/MyDrive/Projects/dataset_line-art'
config.MASK_PATH = '/content/drive/MyDrive/Projects/ground_truth_anime'
config.CKPT_DIR = "/content/drive/MyDrive/rmbg_checkpoints"

print("Config loaded:")
print(f"  IMG_SIZE: {config.IMG_SIZE}")
print(f"  BATCH_SIZE: {config.BATCH_SIZE}")
print(f"  NUM_EPOCHS: {config.NUM_EPOCHS}")

### 5. Load dataset (trapped-ball preprocessing + structural guidance + progressive shuffle)

In [ ]:
from torch.utils.data import DataLoader, random_split
from src.dataset import LineArtDataset

# Enable augmentation for training (line thickness jitter, stroke drop)
dataset = LineArtDataset(config.DATASET_PATH, config.MASK_PATH, config.IMG_SIZE, augment=True)
train_len = int(config.TRAIN_RATIO * len(dataset))
val_len = len(dataset) - train_len
train_set, val_set = random_split(dataset, [train_len, val_len])

train_loader = DataLoader(train_set, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=config.NUM_WORKERS)
val_loader = DataLoader(val_set, batch_size=config.BATCH_SIZE, shuffle=False)

print(f"Train: {train_len} samples, Val: {val_len} samples")

### 6. Visualize sample data

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(4):
    img, mask = dataset[i]
    
    mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
    std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)
    if hasattr(img, 'cpu'):
        img_np = img.cpu().numpy()
    else:
        img_np = img
    img_denorm = img_np * std + mean
    img_denorm = np.clip(np.moveaxis(img_denorm, 0, -1), 0, 1)
    
    axes[0, i].imshow(img_denorm)
    axes[0, i].set_title(f'Image {i+1}')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(mask.squeeze(), cmap='gray')
    axes[1, i].set_title(f'Mask {i+1}')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

### 7. Load model with 3-Phase Freeze Strategy

In [ ]:
import torch
from src.model import load_model
from src.freeze_strategy import BiRefNetFreezeStrategy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load model via src.model to include line-art patch-embed reinitialization
model = load_model(
    model_name=config.MODEL_NAME,
    device=device,
    use_gradient_checkpointing=config.USE_GRADIENT_CHECKPOINTING,
)

# Apply freeze strategy
strategy = BiRefNetFreezeStrategy(model)

# Phase 1: Decoder + Squeeze + PatchEmbed (11.5% params)
strategy.apply_phase_1()
strategy.print_summary("Phase 1: Decoder + Squeeze")

### 8. Differential LR + Optimizer

In [ ]:
# Differential LR: encoder (trainable) = 1e-5, decoder = 5e-4
lr_encoder = 1e-5
lr_decoder = 5e-4

encoder_params = []
decoder_params = []

for name, param in model.named_parameters():
    if param.requires_grad:
        if 'bb.' in name:
            encoder_params.append(param)
        else:
            decoder_params.append(param)

optimizer = torch.optim.AdamW([
    {'params': decoder_params, 'lr': lr_decoder},
    {'params': encoder_params, 'lr': lr_encoder},
], weight_decay=config.WEIGHT_DECAY)

print(f"Model loaded: {config.MODEL_NAME}")
print(f"Device: {device}")
print(f"Encoder trainable params: {sum(p.numel() for p in encoder_params):,}")
print(f"Decoder trainable params: {sum(p.numel() for p in decoder_params):,}")

# Optional: Resume from checkpoint
RESUME_CKPT = None  # Set path like: '/content/drive/MyDrive/rmbg_checkpoints/last_model.pth'
if RESUME_CKPT:
    print(f"\nResume from: {RESUME_CKPT}")

### 9. Test prediction before training

In [ ]:
from src.visualization import visualize_prediction

visualize_prediction(model, dataset, idx=0, device=device)

### 10. Training Phase 1: Decoder + Squeeze (2-3 epochs)

Goal: Learn to translate features to line art masks

In [ ]:
import gc 
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from src.trainer import Trainer

# Note: apply_phase_1() already called in Cell 7
# Training Phase 1
print("=" * 60)
print("PHASE 1: Decoder + Squeeze (2-3 epochs)")
print("=" * 60)
print("Loss: Masked Redrawing (white downweight + edge mask + boundary x100) + 2×Hole")
print("Trainable: ~25M params (11.5%)")

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    device=device,
    ckpt_dir=config.CKPT_DIR,
    num_epochs=3,
    phase=1,
    resume_from=RESUME_CKPT,
    use_boundary_iou=True,
    use_wandb=True,
    wandb_project='rmbg-lineart-ph1'
)
trainer.train(train_loader, val_loader)

### 11. Training Phase 2: + Stage 0-1 + Alternating Stage 2

Now unfreeze low-level features for edge detection

In [ ]:
# Clear GPU memory before Phase 2
import gc
gc.collect()
torch.cuda.empty_cache()

# Phase 2: Open Stage 0-1 + alternating Stage 2 blocks
strategy.apply_phase_2()
strategy.print_summary("Phase 2: Stage 0-1 + Alternating")

# Update optimizer with new trainable params
encoder_params = []
decoder_params = []
for name, param in model.named_parameters():
    if param.requires_grad:
        if 'bb.' in name:
            encoder_params.append(param)
        else:
            decoder_params.append(param)

optimizer = torch.optim.AdamW([
    {'params': decoder_params, 'lr': 2e-4},
    {'params': encoder_params, 'lr': 1e-5},
], weight_decay=config.WEIGHT_DECAY)

In [ ]:
print("PHASE 2: Stage 0-1 + Alternating Stage 2 (5-8 epochs)")
print("-" * 60)
print("Trainable: ~74M params (33.5%)")

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    device=device,
    ckpt_dir=config.CKPT_DIR,
    num_epochs=8,
    phase=2,
    use_boundary_iou=True,
    use_wandb=True,
    wandb_project='rmbg-lineart-ph2'
)
trainer.train(train_loader, val_loader)

### 12. Training Phase 3: Full Fine-tune

Final polish if validation loss still decreasing

In [ ]:
# Clear GPU memory before Phase 3
import gc
gc.collect()
torch.cuda.empty_cache()

# Phase 3: Full fine-tune
strategy.apply_phase_3()
strategy.print_summary("Phase 3: Full Fine-tune")

# Update optimizer (lower LR to reduce overfitting risk in full fine-tuning)
encoder_params = []
decoder_params = []
for name, param in model.named_parameters():
    if param.requires_grad:
        if 'bb.' in name:
            encoder_params.append(param)
        else:
            decoder_params.append(param)

optimizer = torch.optim.AdamW([
    {'params': decoder_params, 'lr': 5e-5},
    {'params': encoder_params, 'lr': 1e-6},
], weight_decay=config.WEIGHT_DECAY)

In [ ]:
print("PHASE 3: Full Fine-tune (5-10 epochs)")
print("=" * 60)
print("Trainable: ~173M params (78.6%)")

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    device=device,
    ckpt_dir=config.CKPT_DIR,
    num_epochs=10,
    phase=3,
    use_boundary_iou=True,
    use_wandb=True,
    wandb_project='rmbg-lineart-ph3'
)
trainer.train(train_loader, val_loader)

### 13. Visualize predictions after training

In [ ]:
!pip install -qqq -r requirements.txt!pip install -U pillow

### 14. Compute final metrics

In [ ]:
from src.losses import compute_all_metrics

model.eval()
metrics_summary = {
    'iou': [],
    'boundary_iou': [],
    'dice': [],
    'precision': [],
    'recall': [],
    'f1': []
}

print("Computing comprehensive metrics...\n")
print(f"{'Metric':<20} {'Mean':>10} {'Std':>10}")
print("-" * 42)

with torch.no_grad():
    for imgs, masks in val_loader:
        imgs = imgs.to(device)
        masks = masks.to(device)
        preds = model(imgs)[-1]
        batch_metrics = compute_all_metrics(preds, masks)
        for k, v in batch_metrics.items():
            metrics_summary[k].append(v.item())

import numpy as np
for k, v in metrics_summary.items():
    mean_val = np.mean(v)
    std_val = np.std(v)
    print(f"{k:<20} {mean_val:>10.4f} {std_val:>10.4f}")

print("\n" + "="*42)
print(f"Best Model from Training:")
print(f"  Best IoU: {trainer.best_iou:.4f}")
print(f"  Best Boundary IoU: {trainer.best_boundary_iou:.4f}")

### 15. Verify checkpoints saved to Drive

In [ ]:
import os

print("Checkpoints in Drive:")
for f in os.listdir(config.CKPT_DIR):
    size = os.path.getsize(os.path.join(config.CKPT_DIR, f)) / 1e6
    print(f"  {f}: {size:.2f} MB")